# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. All data entities—record sets, fields, and columns—are referenced by their `@id`. 

### Dataset Source
The dataset is defined by a Croissant schema, accessible at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{getattr(metadata, 'name', 'Unknown')}\n\n{getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Tip:** In `mlcroissant`, you can inspect record sets and their components programmatically. All entities should be referenced strictly by their `@id` for consistency and reproducibility.

In [ ]:
# List all record sets by @id with their field @id's
record_sets = list(dataset.record_sets)
print("Available record sets and their fields:")
overview = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id} | Name: {getattr(rs, 'name', '(no name)')}")
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields @id's:")
    for fid in field_ids:
        print(f"    - {fid}")
    overview.append({'record_set_id': rs.id, 'field_ids': field_ids})


## 3. Data Extraction
Let's load all data from the main tabular record set into a DataFrame. We'll use the record set and field `@id`s from the previous overview.

**Note:** If multiple record sets are available, they can all be loaded for analysis. Each key in `dataframes` is the record set `@id`.

In [ ]:
# Prepare to extract from all record sets by @id
record_set_ids = [rs['record_set_id'] for rs in overview]
dataframes = {}
for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show list of columns for the main record set (using the first one for demonstration)
main_record_set_id = record_set_ids[0]
print(f"\nColumns (by @id) in record set {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Process and analyze the loaded data. Here, we will:
- Select a numeric field by its `@id` (e.g., age)
- Filter records by a threshold
- Normalize the numeric field
- Optionally, group by another field (e.g., sex or anatomical site) using its field `@id`.

All operations reference fields by their `@id`.

In [ ]:
# Choose numeric field and group field by their @id
# Use dataset exploration above to select a relevant numeric field (by @id)
# Example: age_field_id = '@id-for-age', e.g., 'http://mlcommons.org/croissant/age_at_second_crc' if that's the id
# We'll pick the first numeric field found

main_df = dataframes[main_record_set_id]

# Guess a numeric field by checking datatypes; fallback to first column
numeric_field_id = None
for col in main_df.columns:
    try:
        pd.to_numeric(main_df[col].dropna().iloc[0])
        numeric_field_id = col
        break
    except:
        continue
if numeric_field_id is None:
    numeric_field_id = main_df.columns[0]

print(f"Using numeric field: {numeric_field_id}")

# Set threshold for filtering (e.g., age > 50)
threshold = 50
# Coerce numeric for robust filtering
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally, group by a categorical field (pick if available)
group_field_id = None
for col in main_df.columns:
    if (col != numeric_field_id) and main_df[col].nunique() < main_df.shape[0] // 2:
        group_field_id = col
        break

if group_field_id is not None:
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    )
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize the distribution of the numeric field, and relationship to the group field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If group field selected, boxplot by group
if group_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("No suitable grouping field to visualize.")

## 6. Conclusion
We have loaded and explored the FAIR^2 dataset using `mlcroissant`, referencing all data entities by their `@id` throughout. This notebook:
- Loaded Croissant-metadata driven tabular data into pandas
- Identified record sets, fields, and columns via their globally unique `@id`
- Performed basic exploratory data analysis: filtering and normalization on a selected numeric field, and grouping/categorization where available
- Visualized the core data distributions and relationships.

Further analysis can build from this foundation using the complete, self-describing schema and tooling provided by `mlcroissant`.